# Policy Inference Latency Benchmark
단일 task에서 policy의 실시간성(inference latency)을 측정합니다.

측정 항목:
- **model.predict()** 전체 호출 시간 (preprocessing + inference + postprocessing)
- **실제 forward pass** vs **캐시 반환** 구분 (action_seq_len=10이므로 10스텝 중 1번만 forward)
- **obs preprocessing** 시간
- **env.step()** 시간
- 통계: mean, std, min, max, p50, p95, p99, FPS

In [8]:
import os
import sys
import time
import numpy as np
import torch

# GPU 설정
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.abspath('.'))
if '/Data/pilab/LIBERO' not in sys.path:
    sys.path.insert(0, '/Data/pilab/LIBERO')

from configs.config import create_libero_object_config
from configs.factory import create_model, create_trainer

In [9]:
# 설정
BENCHMARK_TYPE = 'libero_object'
CHECKPOINT_PATH = 'runs/libero_object/2026-03-12/05-41-43/final_model.pth'
DEVICE = 'cuda'
TASK_ID = 0  # 첫 번째 task 사용

In [10]:
# 모델 로드
cfg = create_libero_object_config()
model = create_model(cfg)
trainer = create_trainer(cfg)
model.set_scaler(trainer.scaler)

state_dict = torch.load(CHECKPOINT_PATH, weights_only=True)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()
print(f'Model loaded from {CHECKPOINT_PATH}')

/Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded from runs/libero_object/2026-03-12/05-41-43/final_model.pth


In [11]:
# 환경 설정
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_suite = benchmark.get_benchmark_dict()[BENCHMARK_TYPE]()
task_bddl_file = benchmark_suite.get_task_bddl_file_path(TASK_ID)
file_name = os.path.basename(task_bddl_file).split('.')[0]
task_emb = trainer.trainset.tasks[file_name].to(DEVICE).unsqueeze(0)
init_states = benchmark_suite.get_task_init_states(TASK_ID)

env = OffScreenRenderEnv(
    bddl_file_name=task_bddl_file,
    camera_heights=128,
    camera_widths=128
)
env.seed(0)
env.reset()
obs = env.set_init_state(init_state=init_states[0])

# dummy actions for initial physics
dummy = np.zeros(7)
dummy[-1] = -1.0
for _ in range(5):
    obs, _, _, _ = env.step(dummy)

model.reset()
print(f'Task: {file_name}')
print(f'Action sequence length: {model.action_seq_len}')

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Task: pick_up_the_alphabet_soup_and_place_it_in_the_basket
Action sequence length: 10


In [12]:
# Latency 측정 - 1 에피소드 실행
MAX_STEPS = 600

preprocess_times = []
predict_times = []
predict_forward_times = []   # 실제 forward pass (10스텝에 1번)
predict_cached_times = []    # 캐시된 action 반환 (나머지 9번)
env_step_times = []
total_step_times = []

model.reset()

# GPU warmup
with torch.no_grad():
    for _ in range(3):
        dummy_obs = {
            'agentview_image': torch.randn(1, 1, 3, 128, 128).to(DEVICE),
            'eye_in_hand_image': torch.randn(1, 1, 3, 128, 128).to(DEVICE),
            'lang_emb': task_emb,
            'robot_states': torch.randn(1, 1, 9).to(DEVICE),
        }
        _ = model.predict(dummy_obs)
    model.reset()
    torch.cuda.synchronize()

print('Running episode...')
for step in range(MAX_STEPS):
    t_total_start = time.perf_counter()

    # --- Preprocessing ---
    t0 = time.perf_counter()
    agentview_rgb = torch.from_numpy(obs['agentview_image']).to(DEVICE).float().permute(2, 0, 1).unsqueeze(0).unsqueeze(0) / 255.
    eye_in_hand_rgb = torch.from_numpy(obs['robot0_eye_in_hand_image']).to(DEVICE).float().permute(2, 0, 1).unsqueeze(0).unsqueeze(0) / 255.
    joint_state = obs['robot0_joint_pos']
    gripper_state = obs['robot0_gripper_qpos']
    robot_states = torch.from_numpy(np.concatenate([joint_state, gripper_state], axis=-1)).to(DEVICE).float().unsqueeze(0).unsqueeze(0)
    obs_dict = {
        'agentview_image': agentview_rgb,
        'eye_in_hand_image': eye_in_hand_rgb,
        'lang_emb': task_emb,
        'robot_states': robot_states,
    }
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    preprocess_times.append(t1 - t0)

    # --- Predict ---
    is_forward = (model.rollout_step_counter == 0)
    t2 = time.perf_counter()
    with torch.no_grad():
        action = model.predict(obs_dict).cpu().numpy()
    torch.cuda.synchronize()
    t3 = time.perf_counter()
    predict_times.append(t3 - t2)
    if is_forward:
        predict_forward_times.append(t3 - t2)
    else:
        predict_cached_times.append(t3 - t2)

    # --- Env step ---
    t4 = time.perf_counter()
    obs, r, done, _ = env.step(action)
    t5 = time.perf_counter()
    env_step_times.append(t5 - t4)

    total_step_times.append(t5 - t_total_start)

    if r == 1:
        print(f'SUCCESS at step {step + 1}')
        break

if r != 1:
    print(f'FAILED after {MAX_STEPS} steps')

env.close()
print(f'Total steps: {len(total_step_times)}')

Running episode...
SUCCESS at step 165
Total steps: 165


In [13]:
# 결과 출력
CONTROL_FREQ = 20  # LIBERO 시뮬레이터 control frequency (Hz)
SIM_DT = 1000.0 / CONTROL_FREQ  # 50ms per step

def print_stats(name, times_ms):
    arr = np.array(times_ms)
    print(f'\n  --- {name} ---')
    print(f'    Mean:  {arr.mean():.3f} ms')
    print(f'    Std:   {arr.std():.3f} ms')
    print(f'    Min:   {arr.min():.3f} ms')
    print(f'    Max:   {arr.max():.3f} ms')
    print(f'    P50:   {np.percentile(arr, 50):.3f} ms')
    print(f'    P95:   {np.percentile(arr, 95):.3f} ms')
    print(f'    P99:   {np.percentile(arr, 99):.3f} ms')

# ============================================================
# 1. 에피소드 결과
# ============================================================
print('=' * 70)
print('  1. Episode Result')
print('=' * 70)
total_steps = len(total_step_times)
success = (r == 1)
print(f'  Task:    {file_name}')
print(f'  Result:  {"SUCCESS" if success else "FAILED"}')
print(f'  Steps:   {total_steps} / {MAX_STEPS}')
if success:
    episode_time = sum(total_step_times)
    print(f'  Episode time (wall clock): {episode_time:.2f} s')

# ============================================================
# 2. LIBERO 시뮬레이터 사양
# ============================================================
print('\n' + '=' * 70)
print('  2. LIBERO Simulator Specification')
print('=' * 70)
print(f'  Control frequency:  {CONTROL_FREQ} Hz')
print(f'  Sim dt:             {SIM_DT:.1f} ms (= 1/{CONTROL_FREQ}s)')
print(f'  Max steps/episode:  {MAX_STEPS}')
print(f'  Action dim:         7 (6 DoF + gripper)')
print(f'  Image resolution:   128 x 128')

# ============================================================
# 3. 정책 구조
# ============================================================
print('\n' + '=' * 70)
print('  3. Policy Structure (Action Chunking)')
print('=' * 70)
action_seq_len = model.action_seq_len
num_forward = len(predict_forward_times)
num_cached = len(predict_cached_times)
print(f'  action_seq_len:     {action_seq_len}')
print(f'  Forward passes:     {num_forward} (매 {action_seq_len}스텝에 1회)')
print(f'  Cached returns:     {num_cached} (나머지 {action_seq_len - 1}회)')
print(f'  → 1회 forward로 {action_seq_len}개 action을 생성하고 순차 소비')

# ============================================================
# 4. Latency 상세
# ============================================================
print('\n' + '=' * 70)
print('  4. Latency Breakdown')
print('=' * 70)

preproc_ms = [t * 1000 for t in preprocess_times]
fwd_ms = [t * 1000 for t in predict_forward_times]
cached_ms = [t * 1000 for t in predict_cached_times]
env_ms = [t * 1000 for t in env_step_times]
total_ms = [t * 1000 for t in total_step_times]

print_stats('Obs Preprocessing (numpy→GPU tensor)', preproc_ms)
print_stats('Predict - Forward Pass (실제 inference)', fwd_ms)
print_stats('Predict - Cached Action (캐시 반환)', cached_ms)
print_stats('Env Step (시뮬레이터)', env_ms)
print_stats('Total per Step', total_ms)

# ============================================================
# 5. 실시간성 판정
# ============================================================
print('\n' + '=' * 70)
print('  5. Real-time Feasibility')
print('=' * 70)

fwd_mean = np.mean(fwd_ms)
fwd_p95 = np.percentile(fwd_ms, 95)
fwd_max = np.max(fwd_ms)
cached_mean = np.mean(cached_ms) if cached_ms else 0
preproc_mean = np.mean(preproc_ms)

# Forward step: preprocessing + forward pass (10스텝 중 1번)
forward_step_latency = preproc_mean + fwd_mean
forward_step_p95 = np.mean(preproc_ms) + fwd_p95

# Cached step: preprocessing + cache lookup (10스텝 중 9번)
cached_step_latency = preproc_mean + cached_mean

# Effective: 가중 평균
effective_latency = (forward_step_latency + cached_step_latency * (action_seq_len - 1)) / action_seq_len

print(f'  Simulator requirement:       {SIM_DT:.1f} ms ({CONTROL_FREQ} Hz)')
print()
print(f'  Forward step (1/{action_seq_len}):')
print(f'    Mean latency:              {forward_step_latency:.3f} ms')
print(f'    P95 latency:               {forward_step_p95:.3f} ms')
print(f'    vs sim dt:                 {forward_step_latency:.1f} / {SIM_DT:.1f} ms → {"✓ OK" if forward_step_latency < SIM_DT else "✗ EXCEEDS"}')
print()
print(f'  Cached step ({action_seq_len - 1}/{action_seq_len}):')
print(f'    Mean latency:              {cached_step_latency:.3f} ms')
print(f'    vs sim dt:                 {cached_step_latency:.1f} / {SIM_DT:.1f} ms → {"✓ OK" if cached_step_latency < SIM_DT else "✗ EXCEEDS"}')
print()
print(f'  Effective (가중 평균):')
print(f'    Mean latency:              {effective_latency:.3f} ms')
print(f'    Effective frequency:       {1000.0 / effective_latency:.1f} Hz')
print(f'    vs sim requirement:        {"✓ REAL-TIME FEASIBLE" if effective_latency < SIM_DT else "✗ NOT REAL-TIME"}')

# Worst-case: forward pass의 max
worst_case = np.mean(preproc_ms) + fwd_max
print()
print(f'  Worst case (forward max):')
print(f'    Latency:                   {worst_case:.3f} ms')
print(f'    vs sim dt:                 {worst_case:.1f} / {SIM_DT:.1f} ms → {"✓ OK" if worst_case < SIM_DT else "✗ EXCEEDS"}')

# 시뮬레이터 대비 여유율
margin = (SIM_DT - forward_step_latency) / SIM_DT * 100
print()
print(f'  Forward step margin:         {margin:.1f}% (sim dt 대비 여유)')
print('=' * 70)

  1. Episode Result
  Task:    pick_up_the_alphabet_soup_and_place_it_in_the_basket
  Result:  SUCCESS
  Steps:   165 / 600
  Episode time (wall clock): 2.94 s

  2. LIBERO Simulator Specification
  Control frequency:  20 Hz
  Sim dt:             50.0 ms (= 1/20s)
  Max steps/episode:  600
  Action dim:         7 (6 DoF + gripper)
  Image resolution:   128 x 128

  3. Policy Structure (Action Chunking)
  action_seq_len:     10
  Forward passes:     17 (매 10스텝에 1회)
  Cached returns:     148 (나머지 9회)
  → 1회 forward로 10개 action을 생성하고 순차 소비

  4. Latency Breakdown

  --- Obs Preprocessing (numpy→GPU tensor) ---
    Mean:  0.303 ms
    Std:   0.075 ms
    Min:   0.214 ms
    Max:   0.670 ms
    P50:   0.306 ms
    P95:   0.416 ms
    P99:   0.608 ms

  --- Predict - Forward Pass (실제 inference) ---
    Mean:  23.377 ms
    Std:   1.928 ms
    Min:   21.274 ms
    Max:   30.317 ms
    P50:   22.776 ms
    P95:   26.187 ms
    P99:   29.491 ms

  --- Predict - Cached Action (캐시 반환) ---
    Mea